# Notebook 06 — Item2Vec Embedding Retrieval

**Objective:** Train skip-gram Word2Vec embeddings on time-ordered RetailRocket user-view sequences, generate personalized Top-K item candidates, and compare against Notebook 04 (Popularity) and Notebook 05 (Item-CF) using the **same evaluation protocol**.

**No impression logs:** This is implicit-feedback next-window item retrieval, not CTR estimation. Only events before each cutoff are used for embeddings, popularity, seen-item filters, and user seeds. Future view labels are used **only for evaluation**.

**Execution:** Save this notebook under `notebooks/`, then Run All. Install dependencies if needed: `pip install -U numpy pandas scipy gensim` (restart kernel after installing). Training and neighbor indexing may take several minutes on CPU. No dense user × item matrix is created.

**Protocol:** warm-user future-view pairs, exclude all historically seen items, retain seen/cold-item positives in the primary Recall denominator, exactly matching Notebooks 04–05. Validation is the development split; test is held out for final reporting.

## 1. Read temporal inputs and validate cutoffs

In [4]:
from pathlib import Path
from collections import defaultdict
import json, time
import numpy as np
import pandas as pd

try:
    from gensim.models import Word2Vec
except ImportError as exc:
    raise ImportError('Install gensim: pip install -U gensim scipy numpy; then restart Jupyter kernel') from exc

cwd = Path.cwd().resolve()
if (cwd / 'data' / 'processed').exists():
    ROOT = cwd
elif (cwd.parent / 'data' / 'processed').exists():
    ROOT = cwd.parent
else:
    raise FileNotFoundError(f'Cannot find data/processed from {cwd}')
PROCESSED = ROOT / 'data' / 'processed'
REPORTS = ROOT / 'reports'
MODELS_DIR = ROOT / 'models'
REPORTS.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

names = {
    'val_history': 'train_history.csv',
    'val_labels': 'validation_view_labels_warm_users.csv',
    'test_history': 'test_history.csv',
    'test_labels': 'test_view_labels_warm_users.csv',
}
d = {}
for key, filename in names.items():
    p = PROCESSED / filename
    assert p.is_file(), f'Missing {p}'
    d[key] = pd.read_csv(p)
    print(f'{key}: {len(d[key]):,} rows')

CUTOFFS = {'val': pd.Timestamp('2015-08-01', tz='UTC'),
           'test': pd.Timestamp('2015-09-01', tz='UTC')}
K_VALUES = [10, 20, 50, 100]
MAX_K = max(K_VALUES)
for split in ['val', 'test']:
    h, labels = d[f'{split}_history'], d[f'{split}_labels']
    assert h.timestamp.max() < int(CUTOFFS[split].timestamp() * 1000)
    assert labels.visitorid.isin(h.visitorid).all()
    assert not labels[['visitorid','itemid']].duplicated().any()
    assert h.itemid.notna().all() and h.visitorid.notna().all()
print('PASS: input files, warm-user labels, and temporal boundaries')

val_history: 1,899,029 rows
val_labels: 28,634 rows
test_history: 2,452,391 rows
test_labels: 31,876 rows
PASS: input files, warm-user labels, and temporal boundaries


## 2. Prepare sequential training examples

Word2Vec learns items that appear in similar **local user-session contexts**. RetailRocket does not provide reliable session IDs, so here each user's chronologically ordered views form a proxy sequence. This is a modeling simplification, not a claim about true sessions.

Keep up to 30 most recent view events per user; require at least 2 views for a trainable sequence. No future interactions are included. Token IDs are strings for gensim.

In [5]:
MAX_SEQUENCE_LENGTH = 30
MIN_SEQUENCE_LENGTH = 2

# Deterministic ordering: timestamp plus original row order for equal timestamps.
def make_sequences(history):
    views = history.loc[history.event.eq('view'), ['visitorid','itemid','timestamp']].copy()
    views = views.sort_values(['visitorid','timestamp'], kind='mergesort')
    views = views.groupby('visitorid', sort=False).tail(MAX_SEQUENCE_LENGTH)
    grouped = views.groupby('visitorid', sort=False)['itemid'].agg(list)
    sequences = [[str(int(item)) for item in items]
                 for items in grouped if len(items) >= MIN_SEQUENCE_LENGTH]
    return sequences

sequences = {}
for split in ['val','test']:
    t0 = time.perf_counter()
    sequences[split] = make_sequences(d[f'{split}_history'])
    print(f'{split}: {len(sequences[split]):,} sequences; '
          f'{sum(map(len, sequences[split])):,} tokens; {time.perf_counter()-t0:.1f}s')
assert all(len(seq) >= 2 for split in sequences for seq in sequences[split])

val: 270,872 sequences; 1,029,938 tokens; 14.9s
test: 351,251 sequences; 1,325,864 tokens; 18.0s


## 3. Train time-safe Item2Vec models

Skip-gram with negative sampling (`sg=1`, `negative=5`). `workers=1` makes the training order reproducible for a fixed library version, at the cost of speed. For a faster run set `WORKERS=4` (training may become nondeterministic).

**Important:** The September model is trained only on events before September 1, which includes earlier validation-period behavior. This is appropriate for a time-updated retrieval system. Hyperparameters should be selected using validation, **not** by looking at test performance.

In [6]:
VECTOR_SIZE = 64
WINDOW = 5
MIN_COUNT = 5
EPOCHS = 8
WORKERS = 1
RANDOM_SEED = 42

models = {}
fit_diagnostics = {}
for split in ['val','test']:
    t0 = time.perf_counter()
    model = Word2Vec(
        sentences=sequences[split], vector_size=VECTOR_SIZE,
        window=WINDOW, min_count=MIN_COUNT, sg=1,
        negative=5, epochs=EPOCHS, workers=WORKERS,
        seed=RANDOM_SEED, sorted_vocab=1,
    )
    assert len(model.wv) > MAX_K, 'Too few vocabulary items; lower MIN_COUNT.'
    models[split] = model
    fit_diagnostics[split] = {
        'sequence_count': len(sequences[split]),
        'vocabulary_items': len(model.wv),
        'training_seconds': round(time.perf_counter()-t0, 3),
    }
    model_path = MODELS_DIR / f'notebook06_{split}_item2vec.model'
    model.save(str(model_path))
    print(split, fit_diagnostics[split], 'saved:', model_path)

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


val {'sequence_count': 270872, 'vocabulary_items': 43267, 'training_seconds': 22.745} saved: C:\YR_Python_Code\multi-stage-ads-recommendation\notebooks\models\notebook06_val_item2vec.model


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


test {'sequence_count': 351251, 'vocabulary_items': 52280, 'training_seconds': 30.586} saved: C:\YR_Python_Code\multi-stage-ads-recommendation\notebooks\models\notebook06_test_item2vec.model


## 4. Precompute approximate neighbor candidates in bounded memory

For speed, compute exact cosine similarity **within the learned embedding vocabulary**, but only for up to `MAX_INDEXED_SEEDS` most frequently viewed items. Other seeds fall back to popularity. This is a **restricted seed index**, not full-catalog ANN. Notebook 07 will implement FAISS to remove this seed-index bottleneck.

Vectors are L2-normalized; matrix multiplication computes cosine similarity. Process seed vectors in batches, never create a full vocabulary × vocabulary matrix. `MAX_INDEXED_SEEDS` is a compute budget, not a tuned accuracy parameter.

In [7]:
MAX_INDEXED_SEEDS = 3000
NEIGHBORS_PER_SEED = 80
SIM_BATCH_SIZE = 64
SEED_ITEMS_PER_USER = 10

def fit_popularity(history):
    counts = history.loc[history.event.eq('view'), 'itemid'].value_counts()
    return (counts.rename_axis('itemid').reset_index(name='count')
            .sort_values(['count','itemid'], ascending=[False,True], kind='mergesort')
            .itemid.astype(int).tolist())

def build_neighbor_index(model, popularity):
    tokens = model.wv.index_to_key
    ids = np.array([int(tok) for tok in tokens], dtype=np.int64)
    vec = np.asarray(model.wv.vectors, dtype=np.float32)
    norm = np.linalg.norm(vec, axis=1, keepdims=True)
    vec = np.ascontiguousarray(vec / np.maximum(norm, 1e-12))
    token_to_index = model.wv.key_to_index
    seeds = [int(item) for item in popularity if str(item) in token_to_index][:MAX_INDEXED_SEEDS]
    neighbors = {}
    t0 = time.perf_counter()
    for start in range(0, len(seeds), SIM_BATCH_SIZE):
        batch = seeds[start:start+SIM_BATCH_SIZE]
        indices = np.array([token_to_index[str(item)] for item in batch])
        sims = vec[indices] @ vec.T  # batch x vocabulary only
        for row, seed in enumerate(batch):
            sims[row, indices[row]] = -np.inf
            n = min(NEIGHBORS_PER_SEED, len(ids)-1)
            best = np.argpartition(sims[row], -n)[-n:]
            best = best[np.lexsort((ids[best], -sims[row, best]))]
            neighbors[seed] = [(int(ids[j]), float(sims[row, j])) for j in best
                               if np.isfinite(sims[row,j])]
    return neighbors, {'indexed_seeds':len(seeds),
                       'vocabulary_items':len(ids),
                       'index_build_seconds':round(time.perf_counter()-t0, 3)}

popularity_lists, neighbor_indexes, index_diagnostics = {}, {}, {}
for split in ['val','test']:
    popularity_lists[split] = fit_popularity(d[f'{split}_history'])
    neighbor_indexes[split], index_diagnostics[split] = build_neighbor_index(
        models[split], popularity_lists[split])
    print(split, index_diagnostics[split])

val {'indexed_seeds': 3000, 'vocabulary_items': 43267, 'index_build_seconds': 0.804}
test {'indexed_seeds': 3000, 'vocabulary_items': 52280, 'index_build_seconds': 0.819}


## 5. Generate personalized Top-100 candidates with popularity fallback

Each user contributes up to 10 recent distinct view seeds. Similarity scores from multiple seeds are summed. Seen items (from **all** historical event types) are excluded, matching Notebook 05. Missing or insufficient embedding candidates are filled with the split-specific popularity list.

In [8]:
def user_history_maps(history, users):
    sub = history.loc[history.visitorid.isin(users), ['visitorid','itemid','event','timestamp']]
    seen = sub.groupby('visitorid')['itemid'].agg(set).to_dict()
    views = sub.loc[sub.event.eq('view')].sort_values('timestamp', ascending=False, kind='mergesort')
    views = views.drop_duplicates(['visitorid','itemid'], keep='first')
    seeds = (views.groupby('visitorid', sort=False)['itemid']
             .agg(lambda x: x.head(SEED_ITEMS_PER_USER).astype(int).tolist()).to_dict())
    return seen, seeds

def recommend_item2vec(user, seen_map, seeds_map, neighbors, popularity, k=100):
    seen = seen_map.get(user, set())
    scores = defaultdict(float)
    for seed in seeds_map.get(user, []):
        for item, similarity in neighbors.get(int(seed), []):
            if item not in seen and similarity > 0:
                scores[item] += similarity
    ranked = sorted(scores, key=lambda item: (-scores[item], item))
    recs = ranked[:k]
    personalized_count = len(recs)
    used = set(recs)
    if len(recs) < k:
        for item in popularity:
            if item not in seen and item not in used:
                recs.append(item)
                used.add(item)
                if len(recs) == k:
                    break
    return recs, personalized_count

results = {}
for split in ['val','test']:
    h, labels = d[f'{split}_history'], d[f'{split}_labels']
    users = sorted(int(x) for x in labels.visitorid.unique())
    seen, seeds = user_history_maps(h, users)
    t0 = time.perf_counter()
    recs, personalized_counts = {}, []
    for user in users:
        recs[user], n_personalized = recommend_item2vec(
            user, seen, seeds, neighbor_indexes[split], popularity_lists[split], MAX_K)
        personalized_counts.append(n_personalized)
    elapsed = time.perf_counter() - t0
    assert all(len(v) == MAX_K and len(set(v)) == MAX_K for v in recs.values())
    assert all(not (set(v) & seen.get(u, set())) for u, v in recs.items())
    results[split] = {
        'recs': recs, 'seen': seen, 'users': users,
        'elapsed_s': elapsed,
        'users_with_embedding_candidates': int(sum(x > 0 for x in personalized_counts)),
        'avg_embedding_candidates': float(np.mean(personalized_counts)),
    }
    print(f'{split}: {len(users):,} users; {elapsed:.2f}s batch retrieval; '
          f'{results[split]["users_with_embedding_candidates"]:,} users with embedding candidates; '
          f'{results[split]["avg_embedding_candidates"]:.1f} embedding candidates/user')
# Batch processing time is NOT a valid online p95 latency benchmark.

val: 13,329 users; 0.43s batch retrieval; 4,673 users with embedding candidates; 29.8 embedding candidates/user
test: 15,137 users; 0.46s batch retrieval; 5,255 users with embedding candidates; 29.6 embedding candidates/user


## 6. Evaluate with the **identical** Notebook 05 metrics

- **Micro Recall@K:** all matched future-view pairs / all future-view pairs.
- **Macro Recall@K:** mean of per-user recall.
- **HitRate@K:** fraction of users with at least one future-view hit.
- **CatalogCoverage:** unique recommended items / unique items in historical catalog.

The denominator intentionally includes cold-item and previously seen positive labels, even though these cannot be retrieved under the current policy. Also report the eligible positive pair count.

In [9]:
def evaluate(recs, labels, seen_map, history_items, k_values):
    positives = labels.groupby('visitorid')['itemid'].agg(set).to_dict()
    rows = []
    for k in k_values:
        recalls, hits, found_total, pos_total = [], [], 0, 0
        recommended_items = set()
        for user, truth in positives.items():
            recommended = set(recs[int(user)][:k])
            found = len(recommended & truth)
            recalls.append(found / len(truth))
            hits.append(float(found > 0))
            found_total += found
            pos_total += len(truth)
            recommended_items.update(recommended)
        rows.append({'K': k, 'Recall@K_macro': float(np.mean(recalls)),
                     'Recall@K_micro': found_total / pos_total,
                     'HitRate@K': float(np.mean(hits)),
                     'UniqueRecommendedItems': len(recommended_items),
                     'CatalogCoverage': len(recommended_items) / len(history_items)})
    eligible = sum(1 for u,i in labels[['visitorid','itemid']].itertuples(index=False, name=None)
                   if i in history_items and i not in seen_map.get(int(u), set()))
    return pd.DataFrame(rows), eligible

metrics = {}
for split in ['val','test']:
    m, eligible = evaluate(results[split]['recs'], d[f'{split}_labels'],
                           results[split]['seen'], set(d[f'{split}_history'].itemid.unique()), K_VALUES)
    metrics[split] = m
    results[split]['eligible_pairs'] = eligible
    print(f'\n{split.upper()}: {eligible:,}/{len(d[f"{split}_labels"]):,} eligible positive pairs')
    display(m.round(6))


VAL: 21,984/28,634 eligible positive pairs


,K,Recall@K_macro,Recall@K_micro,HitRate@K,UniqueRecommendedItems,CatalogCoverage
0,10,0.011652,0.010372,0.018681,11741,0.058796
1,20,0.017252,0.015751,0.027684,16932,0.084792
2,50,0.027072,0.026716,0.042614,25317,0.126782
3,100,0.038900,0.038102,0.058819,29974,0.150103



TEST: 24,875/31,876 eligible positive pairs


,K,Recall@K_macro,Recall@K_micro,HitRate@K,UniqueRecommendedItems,CatalogCoverage
0,10,0.010821,0.009004,0.017110,12737,0.057121
1,20,0.016105,0.014086,0.025633,18745,0.084065
2,50,0.025008,0.023246,0.039043,28897,0.129593
3,100,0.036390,0.034164,0.055559,34668,0.155474


## 7. Compare against Popularity and Item-CF (same warm-user protocol)

In [10]:
for split in ['val','test']:
    comparison = metrics[split][['K','Recall@K_micro','HitRate@K','CatalogCoverage']].copy()
    comparison = comparison.rename(columns={c: c+'_Item2Vec' for c in comparison.columns if c != 'K'})
    for nb, name in [('04','Popularity'), ('05','ItemCF')]:
        path = REPORTS / f'notebook{nb}_{split}_{"popularity" if nb=="04" else "itemcf"}_metrics.csv'
        if path.is_file():
            other = pd.read_csv(path)[['K','Recall@K_micro','HitRate@K','CatalogCoverage']]
            other = other.rename(columns={c: c+'_'+name for c in other.columns if c != 'K'})
            comparison = comparison.merge(other, on='K', how='left', validate='one_to_one')
        else:
            print('Optional baseline not found:', path)
    if 'Recall@K_micro_ItemCF' in comparison:
        comparison['Recall_delta_vs_ItemCF'] = (comparison['Recall@K_micro_Item2Vec']
                                                - comparison['Recall@K_micro_ItemCF'])
    print(f'\n{split.upper()} comparison:')
    display(comparison.round(6))
    comparison.to_csv(REPORTS / f'notebook06_{split}_comparison.csv', index=False)

# Do not tune the model using test results. If hyperparameters are changed, compare on validation first.


VAL comparison:


,K,Recall@K_micro_Item2Vec,HitRate@K_Item2Vec,CatalogCoverage_Item2Vec,Recall@K_micro_Popularity,HitRate@K_Popularity,CatalogCoverage_Popularity,Recall@K_micro_ItemCF,HitRate@K_ItemCF,CatalogCoverage_ItemCF,Recall_delta_vs_ItemCF
0,10,0.010372,0.018681,0.058796,0.002968,0.006077,0.000085,0.021443,0.038187,0.244530,-0.011071
1,20,0.015751,0.027684,0.084792,0.005727,0.010728,0.000190,0.032269,0.054018,0.324104,-0.016519
2,50,0.026716,0.042614,0.126782,0.011106,0.019731,0.000431,0.049137,0.078025,0.387493,-0.022421
3,100,0.038102,0.058819,0.150103,0.017636,0.030235,0.000821,0.062373,0.096031,0.405906,-0.024272



TEST comparison:


,K,Recall@K_micro_Item2Vec,HitRate@K_Item2Vec,CatalogCoverage_Item2Vec,Recall@K_micro_Popularity,HitRate@K_Popularity,CatalogCoverage_Popularity,Recall@K_micro_ItemCF,HitRate@K_ItemCF,CatalogCoverage_ItemCF,Recall_delta_vs_ItemCF
0,10,0.009004,0.017110,0.057121,0.002384,0.004955,0.000081,0.019921,0.036863,0.252626,-0.010917
1,20,0.014086,0.025633,0.084065,0.005176,0.010372,0.000188,0.028140,0.049944,0.337870,-0.014054
2,50,0.023246,0.039043,0.129593,0.011325,0.021008,0.000435,0.043544,0.072405,0.407378,-0.020297
3,100,0.034164,0.055559,0.155474,0.017411,0.031314,0.000825,0.054618,0.088591,0.425999,-0.020454


## 8. Save metrics, sample candidates, model metadata, and final assertions

In [11]:
for split in ['val','test']:
    metrics[split].to_csv(REPORTS / f'notebook06_{split}_item2vec_metrics.csv', index=False)
    sample_users = results[split]['users'][:20]
    sample = pd.DataFrame([(u, rank, item) for u in sample_users
                           for rank,item in enumerate(results[split]['recs'][u], start=1)],
                          columns=['visitorid','rank','itemid'])
    sample.to_csv(REPORTS / f'notebook06_{split}_item2vec_sample.csv', index=False)

summary = {
    'method': 'item2vec_skipgram_restricted_seed_cosine_with_popularity_fallback',
    'protocol': 'warm-user future-view pairs; exclude seen; cold/seen labels remain in denominator',
    'cutoffs_utc': {'validation':'2015-08-01', 'test':'2015-09-01'},
    'config': {
        'max_sequence_length': MAX_SEQUENCE_LENGTH,
        'vector_size': VECTOR_SIZE, 'window': WINDOW, 'min_count': MIN_COUNT,
        'epochs': EPOCHS, 'workers': WORKERS, 'random_seed': RANDOM_SEED,
        'max_indexed_seeds': MAX_INDEXED_SEEDS,
        'neighbors_per_seed': NEIGHBORS_PER_SEED,
        'seed_items_per_user': SEED_ITEMS_PER_USER, 'max_k': MAX_K,
    },
    'fit_diagnostics': fit_diagnostics,
    'index_diagnostics': index_diagnostics,
}
for split, label in [('val','validation'),('test','test')]:
    summary[label] = {
        'users': len(results[split]['users']),
        'positive_pairs': len(d[f'{split}_labels']),
        'eligible_pairs': results[split]['eligible_pairs'],
        'batch_seconds': results[split]['elapsed_s'],
        'users_with_embedding_candidates': results[split]['users_with_embedding_candidates'],
        'avg_embedding_candidates': results[split]['avg_embedding_candidates'],
        'metrics': metrics[split].to_dict(orient='records'),
    }

out = REPORTS / 'notebook06_item2vec_retrieval.json'
out.write_text(json.dumps(summary, indent=2), encoding='utf-8')
assert all(results[s]['eligible_pairs'] <= len(d[f'{s}_labels']) for s in ['val','test'])
assert all(metrics[s]['Recall@K_micro'].between(0,1).all() for s in ['val','test'])
assert all(metrics[s]['K'].tolist() == K_VALUES for s in ['val','test'])
print('Saved:', out)
print('PASS: temporal cutoffs, warm users, candidate uniqueness/seen filtering, and metric exports')

Saved: C:\YR_Python_Code\multi-stage-ads-recommendation\notebooks\reports\notebook06_item2vec_retrieval.json
PASS: temporal cutoffs, warm users, candidate uniqueness/seen filtering, and metric exports


## 9. Interpretation and next step

- Compare **validation** Recall@50 / Recall@100 to Notebook 04 and Notebook 05. More complex embeddings are **not guaranteed** to beat simple baselines.
- Track `users_with_embedding_candidates` and `avg_embedding_candidates` to understand the limited seed index.
- `CatalogCoverage` is distinct from Recall; broad catalog exposure can coexist with weak accuracy.
- **Next: Notebook 07 — FAISS ANN Retrieval.** Replace the restricted seed index with a full-vocabulary approximate nearest-neighbor index; preserve the same temporal split, filtering, fallback, and evaluation protocol.